In [1]:
%%capture
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from google.colab import files

In [2]:
files.upload()
data = pd.read_csv("data_SWRC.csv", sep=";")

Saving data_SWRC.csv to data_SWRC.csv


In [3]:
dataset = data.values
min_alpha, max_alpha = min(dataset[: ,11:12])[0], max(dataset[: ,11:12])[0]
min_n, max_n = min(dataset[: ,12:13])[0], max(dataset[: ,12:13])[0]
min_thetar, max_thetar = min(dataset[: ,13:14])[0], max(dataset[: ,13:14])[0]
min_thetas, max_thetas = min(dataset[: ,14:15])[0], max(dataset[: ,14:15])[0]
scaler_target_min, scaler_target_max = -1, 1
scaler = MinMaxScaler(feature_range=(scaler_target_min, scaler_target_max))
dataset = scaler.fit_transform(dataset)
np.take(dataset,np.random.permutation(dataset.shape[0]),axis=0,out=dataset)
total_size = len(dataset)
train_size = round(total_size*0.75)
test_size = round(total_size*0.2)
learning_size = train_size + test_size
validation_size = round(total_size*0.05)
train_dataset = dataset[0:train_size]
test_dataset = dataset[train_size:learning_size]
validation_dataset = dataset[learning_size:total_size]
input_train, output_train = train_dataset[:,0:11], train_dataset[: ,11:15]
input_test, output_test = test_dataset[:,0:11], test_dataset[: ,11:15]
input_validation, output_validation = validation_dataset[:,0:11], validation_dataset[: ,11:15]

In [4]:
def build_model():
  model = keras.Sequential([
    layers.Dense(800, activation="tanh", input_shape=[11]),
    layers.Dense(640, activation="tanh"),
    layers.Dense(160, activation="tanh"),
    layers.Dense(40, activation="tanh"),
    layers.Dense(4, activation="tanh")])
  optimizer = tf.keras.optimizers.Adamax(learning_rate=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-07, name="Adamax")
  model.compile(loss="mse", optimizer=optimizer, metrics=["mae", "mape", "msle", "logcosh"])
  return model
model = build_model()
epoch = 150
history = model.fit(input_train, output_train, epochs=epoch, validation_data=(input_test, output_test), verbose=0)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 800)               9600      
                                                                 
 dense_1 (Dense)             (None, 640)               512640    
                                                                 
 dense_2 (Dense)             (None, 160)               102560    
                                                                 
 dense_3 (Dense)             (None, 40)                6440      
                                                                 
 dense_4 (Dense)             (None, 4)                 164       
                                                                 
Total params: 631404 (2.41 MB)
Trainable params: 631404 (2.41 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
